# Image–Label Pair Cleaner

Takes a folder of **images** and a folder of **labels**, keeps only the files that have a matching partner (same filename, ignoring extension), and saves the cleaned pairs into **new folders** on your computer.

Your original folders are **never modified**: matched files are copied to the output folder.

**How to use:** edit the paths in Step 1, then run all cells top to bottom.

## Step 1: Settings

In [ ]:
from pathlib import Path

# ---- EDIT THESE ----
IMAGES_DIR = Path(r"C:\path\to\images")    # folder with your images
LABELS_DIR = Path(r"C:\path\to\labels")    # folder with your labels
OUTPUT_DIR = Path(r"C:\path\to\cleaned")   # new folder; will contain /images and /labels

# Which file types count as images / labels
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
LABEL_EXTS = {".txt", ".xml", ".json", ".csv"}   # YOLO .txt, Pascal VOC .xml, COCO-style .json, etc.

SEARCH_SUBFOLDERS = False   # True = also look inside subfolders
DRY_RUN = False             # True = only report what would happen, copy nothing

## Step 2: Scan folders and find matches

In [ ]:
def collect(folder, exts, recursive):
    """Return {stem: [paths]} for files in folder with allowed extensions."""
    folder = Path(folder)
    if not folder.is_dir():
        raise FileNotFoundError(f"Folder not found: {folder}")
    files = folder.rglob("*") if recursive else folder.iterdir()
    found = {}
    for f in sorted(files):
        if f.is_file() and f.suffix.lower() in exts:
            found.setdefault(f.stem, []).append(f)
    return found

images = collect(IMAGES_DIR, IMAGE_EXTS, SEARCH_SUBFOLDERS)
labels = collect(LABELS_DIR, LABEL_EXTS, SEARCH_SUBFOLDERS)

# Warn about duplicate names (e.g. cat.jpg and cat.png), since only one can pair with a label
for name, d in (("images", images), ("labels", labels)):
    dupes = {s: p for s, p in d.items() if len(p) > 1}
    if dupes:
        print(f"WARNING: {len(dupes)} duplicate name(s) in {name}; only the first of each will be kept:")
        for s, p in list(dupes.items())[:5]:
            print("   ", [x.name for x in p])

matched = sorted(set(images) & set(labels))
images_only = sorted(set(images) - set(labels))
labels_only = sorted(set(labels) - set(images))

print(f"Images found:            {len(images)}")
print(f"Labels found:            {len(labels)}")
print(f"Matched pairs (kept):    {len(matched)}")
print(f"Images without a label:  {len(images_only)}  (will be dropped)")
print(f"Labels without an image: {len(labels_only)}  (will be dropped)")

if images_only:
    print("\nExample images with no label:", [images[s][0].name for s in images_only[:5]])
if labels_only:
    print("Example labels with no image:", [labels[s][0].name for s in labels_only[:5]])

## Step 3: Save the cleaned folders

Copies matched pairs to `OUTPUT_DIR/images` and `OUTPUT_DIR/labels`.

In [ ]:
import shutil

out_images = OUTPUT_DIR / "images"
out_labels = OUTPUT_DIR / "labels"

if DRY_RUN:
    print(f"DRY RUN: would copy {len(matched)} pairs to {OUTPUT_DIR}")
else:
    if matched and (out_images.exists() or out_labels.exists()):
        print("Note: output folders already exist; files with the same name will be overwritten.")
    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    for stem in matched:
        shutil.copy2(images[stem][0], out_images / images[stem][0].name)
        shutil.copy2(labels[stem][0], out_labels / labels[stem][0].name)

    print(f"Done. Copied {len(matched)} pairs to:")
    print("  ", out_images.resolve())
    print("  ", out_labels.resolve())

## Step 4: Verify the result

In [ ]:
if not DRY_RUN:
    new_img = {p.stem for p in out_images.iterdir() if p.is_file()}
    new_lbl = {p.stem for p in out_labels.iterdir() if p.is_file()}
    print(f"Output images: {len(new_img)} | Output labels: {len(new_lbl)}")
    if new_img == new_lbl:
        print("Every image has a label and every label has an image.")
    else:
        print("Mismatch found!")
        print("Images without labels:", sorted(new_img - new_lbl)[:10])
        print("Labels without images:", sorted(new_lbl - new_img)[:10])

## Optional: Zip / download (for Google Colab or remote Jupyter)

If you run this notebook locally, the folders are already on your computer and you can skip this. If you run it in **Google Colab**, the output lives on Colab's temporary storage, so run this cell to download it as a zip.

In [ ]:
zip_path = shutil.make_archive(str(OUTPUT_DIR), "zip", root_dir=OUTPUT_DIR)
print("Created:", zip_path)

try:
    from google.colab import files   # only works inside Colab
    files.download(zip_path)
except ImportError:
    print("Not running in Colab; the zip is saved next to your output folder.")